In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 06. MoE Router | 混合专家架构: 稀疏路由与负载均衡 (MoE)

### 核心思想与痛点

> **Dense (稠密) 模型的痛点：**
> 在标准的 Transformer 中，每一个 Token 都必须经过全网络的所有参数（比如 70B 的 LLaMA）。这导致随着模型变大，推理和训练的计算量呈线性爆炸。
> 
> **MoE 的破局：稀疏激活 (Sparse Activation)**
> 将原来大规模的 MLP 层，切分成 $N$ 个小型的独立 MLP（称为 Expert，专家）。
> 对于每一个输入的 Token，通过一个非常轻量的 Router (门控网络) 决定它该去请教哪 $K$ 个专家（通常 $K=2$）。
> 这样，即便总参数量有 8x7B=56B，实际每个 Token 只激活了 2x7B=14B 的参数。**计算量骤降，而知识容量剧增。**

###  核心数学机制：Top-K Routing

**1. 门控网络 (Gating / Router)：**
给定输入 Token 的特征 $x \in \mathbb{R}^d$，我们用一个线性层将其映射到各个专家的打分：
$$ h = x W_{gate} \quad (h \in \mathbb{R}^E) $$
其中 $E$ 是专家总数（如 8）。

**2. 全局归一化与 Top-K 选择 (The Softmax Trap)：**
传统初学者容易犯的错误是先选 Top-K 的 Logits，再做 Softmax。正确的工业级做法（如 Mixtral 8x7B）必须是先做全局 Softmax：
$$ p = \text{Softmax}(h) \quad (p \in \mathbb{R}^E) $$
为了保持稀疏性，我们提取其中概率最高的 $K$ 个专家：
$$ p_{topk}, idx_{topk} = \text{TopK}(p, K) $$

**3. 局部重归一化 (Re-normalize)：**
由于截取了部分概率，剩下的 $K$ 个概率之和不再为 1。为了稳定梯度的尺度，必须按比例将其重新归一化：
$$ w_i = \frac{p_i}{\sum_{j \in TopK} p_j} $$

**4. 最终输出融合：**
Token 经过这 $K$ 个专家的计算后，按最新权重加权求和：
$$ y = \sum_{i \in TopK} w_i \cdot \text{Expert}_{idx_i}(x) $$

In [2]:
class TopKRouter(nn.Module):
    def __init__(self, hidden_size: int, num_experts: int, top_k: int):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        
        # 定义门控层，将隐藏状态映射到专家数量的得分
        self.gate = nn.Linear(hidden_size, num_experts, bias=False)

    def forward(self, hidden_states: torch.Tensor):
        """
        Args:
            hidden_states: [batch_size, seq_len, hidden_size]
        Returns:
            routing_weights: 形状 [batch_size * seq_len, top_k]，表示选中的专家的权重 (重归一化后)
            selected_experts: 形状 [batch_size * seq_len, top_k]，表示选中的专家索引
        """
        batch_size, seq_len, hidden_size = hidden_states.shape
        # 展平输入
        hidden_states = hidden_states.view(-1, hidden_size)
        
        # 1. 计算 logits 得分
        router_logits = self.gate(hidden_states)
        
        # ==========================================
        # TODO 1: 对全量 Logits 进行 Softmax 获取所有专家的概率分布
        # 提示: 强制使用 FP32 以防止精度溢出 (router_logits.float())
        # ==========================================
        routing_probs = F.softmax(router_logits.float(), dim=-1)
        
        # ==========================================
        # TODO 2: 从概率分布中截取 Top-K 最大的概率 (routing_weights) 及其索引 (selected_experts)
        # ==========================================
        routing_weights, selected_experts = torch.topk(routing_probs, self.top_k, dim=-1)
        
        # ==========================================
        # TODO 3: 对截取后的 routing_weights 进行重归一化 (Re-normalize)
        # 提示: 让这 K 个专家的概率按比例放大，使其加和等于 1
        # ==========================================
        routing_weights = routing_weights / routing_weights.sum(dim=-1, keepdim=True)                                                                                             

        # 恢复到原始数据类型
        routing_weights = routing_weights.to(hidden_states.dtype)
        
        return routing_weights, selected_experts

# 为了验证 Router 能正确工作，我们写一个极简的 MoE 聚合层
class SparseMoEBlock(nn.Module):
    def __init__(self, hidden_size: int, num_experts: int, top_k: int):
        super().__init__()
        self.router = TopKRouter(hidden_size, num_experts, top_k)
        # 极简模拟 Expert (真实的 Expert 通常是 SwiGLU MLP)
        self.experts = nn.ModuleList([nn.Linear(hidden_size, hidden_size) for _ in range(num_experts)])
        
    def forward(self, hidden_states: torch.Tensor):
        batch_size, seq_len, hidden_size = hidden_states.shape
        routing_weights, selected_experts = self.router(hidden_states)
        
        final_hidden_states = torch.zeros(
            (batch_size * seq_len, hidden_size), 
            dtype=hidden_states.dtype, 
            device=hidden_states.device
        )
        flat_hidden_states = hidden_states.view(-1, hidden_size)
        
        # 工业界(vLLM/Megatron)会通过 Token Sorting (索引排序) 汇聚同专家的Token，
        # 这里为便于理解核心算法逻辑，使用 For 循环遍历被选中的 Expert
        for expert_idx, expert in enumerate(self.experts):
            token_idx, kth_expert = torch.where(selected_experts == expert_idx)
            if token_idx.shape[0] > 0:
                current_state = flat_hidden_states[token_idx]
                current_output = expert(current_state)
                current_weight = routing_weights[token_idx, kth_expert].unsqueeze(-1)
                final_hidden_states[token_idx] += current_output * current_weight
                
        return final_hidden_states.view(batch_size, seq_len, hidden_size)


**工程优化要点**

- **稀疏激活的本质**：MoE 的核心价值在于"大容量、低激活"。通过 Top-K 路由，每个 Token 只激活少数专家（通常 K=2），使得 56B 参数的模型实际计算量仅相当于 14B 的稠密模型。
- **高效聚合**：代码中的 `SparseMoEBlock` 使用 For 循环遍历专家是为了便于理解。工业界框架（vLLM、Megatron-LM）会使用 Token Sorting（按专家索引排序）将去往同一专家的 Token 汇聚成批次，一次性计算以提升 GPU 利用率。
